In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder.appName("Day06_DQ").getOrCreate()

# Simulate DQ Engine Logic
def validate_and_route(df, rules):
    error_condition = lit(False)
    
    for column, column_rules in rules.items():
        if "not_null" in column_rules:
            error_condition = error_condition | col(column).isNull()
        if "valid_vitals" in column_rules:
            # Example: Heart rate must be between 30 and 250
            error_condition = error_condition | (col(column) < 30) | (col(column) > 250)
            
    invalid_df = df.filter(error_condition).withColumn("dq_error_manifest", lit("Failed Clinical Plausibility or Schema bounds"))
    valid_df = df.filter(~error_condition)
    
    return valid_df, invalid_df

data = [
    ("obs-1", "HeartRate", 85), # Valid
    ("obs-2", "HeartRate", None), # Invalid (Null)
    ("obs-3", "HeartRate", 999), # Invalid (Out of bounds)
    ("obs-4", "HeartRate", 60)  # Valid
]
df = spark.createDataFrame(data, ["observation_id", "measurement_type", "value_as_number"])

rules = {
    "value_as_number": ["not_null", "valid_vitals"]
}

valid_df, invalid_df = validate_and_route(df, rules)

print("VALID RECORDS (Targeted for Silver OMOP):")
valid_df.show()

print("QUARANTINED RECORDS (Routed to DLQ):")
invalid_df.show(truncate=False)
